# MMUbiPred–Context Residual Hybrid: manuscript visualizations

This notebook generates five post hoc visualization figures for the frozen residual hybrid: t-SNE, ROC curves, precision–recall curves, confusion-matrix comparison, and five-fold training/validation curves. Saved probabilities are reused for every performance plot. One frozen inference pass is performed only when hidden representations required by t-SNE have not already been cached. Every figure is displayed after generation and exported as PDF, SVG, and 600-DPI PNG. No training, threshold selection, model selection, or metric optimization occurs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Repository and model access

Private-repository access uses a fine-grained read-only GitHub token stored in Colab Secrets as `GITHUB_TOKEN`. An optional `HF_TOKEN` supports authenticated ESM-2 retrieval. The Hugging Face cache remains on Drive.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN, HF_TOKEN = secret('GITHUB_TOKEN'), secret('HF_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/content/drive/MyDrive/MMUbiPred-research/huggingface-cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import hashlib, json, numpy as np, torch
from IPython.display import display
import matplotlib.pyplot as plt

print('Device available:', 'cuda' if torch.cuda.is_available() else 'cpu')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

## Frozen-artifact preflight

The visualization uses the completed five-fold development run, frozen five-epoch local refit, frozen seven-epoch context refit, matched historical-test probabilities, and historical-test UniProt cache. The preflight reports every required path before feature extraction.

In [ ]:
CONFIG = Path('experiment/configs/mmubipred_context_residual_v1.json')
DEVELOPMENT_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-v1')
CONTEXT_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-only-final-v1-seed42')
HYBRID_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-full-refit-exploratory-v1')
TEST_SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/context/uniprot_test_sequences.json')
VISUALIZATION_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/visualizations/residual-hybrid-v1')
FIGURE_DIR = VISUALIZATION_DIR / 'figures'
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

required = {
    'development summary': DEVELOPMENT_RUN_DIR / 'stabilized_summary.json',
    'local checkpoint': HYBRID_RUN_DIR / 'best.pt',
    'hybrid metrics': HYBRID_RUN_DIR / 'residual_hybrid_test_metrics.json',
    'hybrid predictions': HYBRID_RUN_DIR / 'residual_hybrid_test_predictions.npz',
    'context checkpoint': CONTEXT_RUN_DIR / 'best.pt',
    'context metrics': CONTEXT_RUN_DIR / 'context_test_metrics.json',
    'matched comparison': CONTEXT_RUN_DIR / 'matched_mmubipred_comparison.json',
    'test sequence cache': TEST_SEQUENCE_CACHE,
}
for role, path in required.items():
    print(f'{role:24s}', 'FOUND' if path.exists() else 'MISSING', path)
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing frozen artifacts:\n' + '\n'.join(missing))

## Frozen hidden-representation extraction

The extraction is skipped when a completed cache exists. Otherwise, a GPU runtime is required for the LoRA-ESM-2 pass. The script verifies that recomputed probabilities reproduce the saved frozen probabilities before accepting either hidden matrix.

In [ ]:
REPRESENTATIONS_PATH = VISUALIZATION_DIR / 'matched_hidden_representations.npz'
REPRESENTATION_MANIFEST_PATH = VISUALIZATION_DIR / 'representation_manifest.json'

if REPRESENTATIONS_PATH.exists() and REPRESENTATION_MANIFEST_PATH.exists():
    print('Existing frozen representation cache found; inference was not repeated.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('A GPU runtime is required for the one-time ESM-2 representation extraction.')
    run_live([
        sys.executable, '-u',
        'experiment/scripts/extract_residual_hybrid_representations.py',
        '--config', CONFIG,
        '--hybrid-run-dir', HYBRID_RUN_DIR,
        '--context-run-dir', CONTEXT_RUN_DIR,
        '--test-sequence-cache', TEST_SEQUENCE_CACHE,
        '--output-dir', VISUALIZATION_DIR,
        '--batch-size', '32',
        '--num-workers', '0',
        '--allow-historical-test-visualization',
    ])

representation_manifest = json.loads(REPRESENTATION_MANIFEST_PATH.read_text())
if sha256_file(REPRESENTATIONS_PATH) != representation_manifest['output']['sha256']:
    raise RuntimeError('The frozen representation-cache checksum changed.')
display(representation_manifest)

## Load aligned visualization data

In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'experiment' / 'src'))
from ubipred.visualization import (
    compute_tsne,
    plot_confusion_matrix_comparison,
    plot_precision_recall_curves,
    plot_roc_curves,
    plot_training_validation_curves,
    plot_tsne_comparison,
    save_figure,
    stratified_visualization_indices,
)

with np.load(REPRESENTATIONS_PATH, allow_pickle=False) as saved:
    dataset_indices = saved['dataset_indices'].astype(np.int64)
    labels = saved['labels'].astype(np.int64)
    local_representations = saved['local_representations'].astype(np.float32)
    context_representations = saved['context_representations'].astype(np.float32)
    probabilities = {
        'Exact MMUbiPred': saved['paper_probabilities'].astype(np.float64),
        'Local expert': saved['local_probabilities'].astype(np.float64),
        'Context expert': saved['context_probabilities'].astype(np.float64),
        'Residual hybrid': saved['residual_probabilities'].astype(np.float64),
    }

assert local_representations.shape == (len(labels), 6)
assert context_representations.shape == (len(labels), 256)
assert all(values.shape == labels.shape for values in probabilities.values())
display({
    'cohort_size': len(labels),
    'negative_sites': int(np.sum(labels == 0)),
    'positive_sites': int(np.sum(labels == 1)),
    'local_representation_shape': local_representations.shape,
    'context_representation_shape': context_representations.shape,
})

## Figure 1 — t-SNE representation comparison

The same deterministic class-balanced subset is used in both panels. t-SNE coordinates are cached because the nonlinear embedding is computationally expensive. This figure is qualitative and is not used for model selection or performance claims.

In [ ]:
TSNE_PATH = VISUALIZATION_DIR / 'tsne_coordinates_seed42.npz'
TSNE_MAX_POINTS = 6000
TSNE_SEED = 42
TSNE_PERPLEXITY = 30.0
TSNE_ITERATIONS = 1500

sample_indices = stratified_visualization_indices(labels, TSNE_MAX_POINTS, TSNE_SEED)
if TSNE_PATH.exists():
    with np.load(TSNE_PATH, allow_pickle=False) as cached:
        cached_indices = cached['sample_indices'].astype(np.int64)
        if not np.array_equal(cached_indices, sample_indices):
            raise RuntimeError('Cached t-SNE sample indices do not match the frozen configuration.')
        local_tsne = cached['local_tsne']
        context_tsne = cached['context_tsne']
        if int(cached['seed']) != TSNE_SEED or float(cached['perplexity']) != TSNE_PERPLEXITY or int(cached['iterations']) != TSNE_ITERATIONS:
            raise RuntimeError('Cached t-SNE parameters do not match the frozen configuration.')
        if str(cached['representations_sha256']) != sha256_file(REPRESENTATIONS_PATH):
            raise RuntimeError('Cached t-SNE coordinates use a different representation artifact.')
    print('Existing t-SNE coordinates reused:', TSNE_PATH)
else:
    print('Computing local-expert t-SNE...', flush=True)
    local_tsne = compute_tsne(
        local_representations[sample_indices],
        seed=TSNE_SEED, perplexity=TSNE_PERPLEXITY, iterations=TSNE_ITERATIONS,
    )
    print('Computing context-expert t-SNE...', flush=True)
    context_tsne = compute_tsne(
        context_representations[sample_indices],
        seed=TSNE_SEED, perplexity=TSNE_PERPLEXITY, iterations=TSNE_ITERATIONS,
    )
    np.savez_compressed(
        TSNE_PATH,
        sample_indices=sample_indices,
        dataset_indices=dataset_indices[sample_indices],
        labels=labels[sample_indices],
        local_tsne=local_tsne,
        context_tsne=context_tsne,
        seed=np.asarray(TSNE_SEED),
        perplexity=np.asarray(TSNE_PERPLEXITY),
        iterations=np.asarray(TSNE_ITERATIONS),
        representations_sha256=np.asarray(sha256_file(REPRESENTATIONS_PATH)),
    )
display({
    'visualized_sites': len(sample_indices),
    'negative_sites': int(np.sum(labels[sample_indices] == 0)),
    'positive_sites': int(np.sum(labels[sample_indices] == 1)),
    'seed': TSNE_SEED,
    'perplexity': TSNE_PERPLEXITY,
    'iterations': TSNE_ITERATIONS,
})

In [ ]:
figure = plot_tsne_comparison(local_tsne, context_tsne, labels[sample_indices])
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_01_tsne_representations')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Figure 2 — ROC curves

All four models are compared on the identical 12,288-site matched cohort using their saved positive-class probabilities.

In [ ]:
figure = plot_roc_curves(labels, probabilities)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_02_roc_curves')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Figure 3 — Precision–recall curves

The horizontal reference line denotes positive-class prevalence in the matched cohort.

In [ ]:
figure = plot_precision_recall_curves(labels, probabilities)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_03_precision_recall_curves')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Figure 4 — Confusion-matrix comparison

Counts and row-normalized percentages are reported at the frozen threshold 0.5. Row normalization separates sensitivity from specificity despite class imbalance.

In [ ]:
figure = plot_confusion_matrix_comparison(labels, probabilities, threshold=0.5)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_04_confusion_matrices')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Figure 5 — Five-fold training and validation curves

The local panels use the uniformly stabilized initialization selected within each fold. The context panels use the original unchanged context histories. Validation MCC is evaluated at threshold 0.5. These are development curves; the final five-epoch and seven-epoch full-data refits had no validation sets.

In [ ]:
local_histories, context_histories, selected_local_seeds = [], [], []
for fold in range(5):
    fold_dir = DEVELOPMENT_RUN_DIR / 'folds' / f'fold_{fold}'
    selection_path = fold_dir / 'local_stabilized' / 'selection.json'
    context_history_path = fold_dir / 'context' / 'development_history.json'
    if not selection_path.exists() or not context_history_path.exists():
        raise FileNotFoundError(f'Incomplete fold history: {fold_dir}')
    selection = json.loads(selection_path.read_text())
    selected_seed = int(selection['selected_initialization_seed'])
    local_history_path = (
        fold_dir / 'local_stabilized' / 'candidates' / f'seed_{selected_seed}' / 'development_history.json'
    )
    if not local_history_path.exists():
        raise FileNotFoundError(f'Missing selected local history: {local_history_path}')
    selected_local_seeds.append(selected_seed)
    local_histories.append(json.loads(local_history_path.read_text()))
    context_histories.append(json.loads(context_history_path.read_text()))

histories = {
    'MMUbiPred-compatible local expert': local_histories,
    'LoRA-ESM-2 context expert': context_histories,
}
display({
    'folds': list(range(5)),
    'selected_local_initialization_seeds': selected_local_seeds,
    'local_epochs_completed': [len(history) for history in local_histories],
    'context_epochs_completed': [len(history) for history in context_histories],
})

In [ ]:
figure = plot_training_validation_curves(histories)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_05_training_validation_curves')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Output inventory

In [ ]:
generated = sorted(path for path in FIGURE_DIR.iterdir() if path.is_file())
expected_stems = {
    'figure_01_tsne_representations',
    'figure_02_roc_curves',
    'figure_03_precision_recall_curves',
    'figure_04_confusion_matrices',
    'figure_05_training_validation_curves',
}
expected_files = {f'{stem}.{suffix}' for stem in expected_stems for suffix in ('pdf', 'svg', 'png')}
observed_files = {path.name for path in generated}
if not expected_files.issubset(observed_files):
    raise RuntimeError(f'Missing figure exports: {sorted(expected_files - observed_files)}')
figure_manifest = {
    'status': 'five manuscript visualization figures generated',
    'analysis_role': 'post hoc visualization; no model or threshold selection',
    'cohort_records': len(labels),
    'matched_test_indices_sha256': representation_manifest['cohort']['matched_indices_sha256'],
    'representation_artifact_sha256': sha256_file(REPRESENTATIONS_PATH),
    'tsne': {
        'sample_size': len(sample_indices),
        'seed': TSNE_SEED,
        'perplexity': TSNE_PERPLEXITY,
        'iterations': TSNE_ITERATIONS,
        'coordinates_sha256': sha256_file(TSNE_PATH),
    },
    'classification_threshold': 0.5,
    'figures': {path.name: sha256_file(path) for path in generated},
}
FIGURE_MANIFEST_PATH = VISUALIZATION_DIR / 'figure_manifest.json'
FIGURE_MANIFEST_PATH.write_text(json.dumps(figure_manifest, indent=2, sort_keys=True) + '\n')
display({
    'figure_directory': str(FIGURE_DIR),
    'generated_files': [path.name for path in generated],
    'representation_cache': str(REPRESENTATIONS_PATH),
    'tsne_cache': str(TSNE_PATH),
    'figure_manifest': str(FIGURE_MANIFEST_PATH),
    'interpretation': figure_manifest['analysis_role'],
})